# Retail Sales Analytics and Dashboard Design

This notebook demonstrates exploratory analysis and visualization
of retail sales data, serving as the foundation for an interactive
business dashboard.



## Loading and Inspecting the Raw Data

In [ ]:
import pandas as pd
filename = '11-ralphs_sales.csv.gz'
raw  = pd.read_csv(filename)


,Geography,Time,Product,Dollar Sales,Unit Sales
0,Kroger Ralphs-RMA - Groc,"1 Week Ending Dec 31, 2017",CHESTERS CORN & POTATO SNACK FLAMIN HOT BAG 5....,$9763.0,4882
1,Kroger Ralphs-RMA - Groc,"1 Week Ending Dec 31, 2017",CHESTERS CORN & POTATO SNACK FLAMIN HOT BAG 1....,$77.5,155
2,Kroger Ralphs-RMA - Groc,"1 Week Ending Dec 31, 2017",CHESTERS CORN & POTATO SNACK FLAMIN HOT BAG 4 ...,$2834.13,1677
3,Kroger Ralphs-RMA - Groc,"1 Week Ending Dec 31, 2017",SABRITONES WHEAT SNACK CHILI & LIME BAG 4.25 O...,$446.0,223
4,Kroger Ralphs-RMA - Groc,"1 Week Ending Dec 31, 2017",CHEETOS CHEESE SNACK CHEESE 50% LESS FAT BAG 7...,$1872.3,572
...,...,...,...,...,...
14124,Kroger Ralphs-RMA - Groc,"1 Week Ending Aug 26, 2018",ALL FRITO LAY PRODUCTS ASTSS SALTED SNACKS ASS...,$7712.3,1070
14125,Kroger Ralphs-RMA - Groc,"1 Week Ending Aug 26, 2018",ALL FRITO LAY PRODUCTS ASTSS SALTED SNACKS ASS...,$34487.06,4794
14126,Kroger Ralphs-RMA - Groc,"1 Week Ending Aug 26, 2018",ALL FRITO LAY PRODUCTS ASTSS SALTED SNACKS ASS...,$2996.84,416
14127,Kroger Ralphs-RMA - Groc,"1 Week Ending Aug 26, 2018",ALL FRITO LAY PRODUCTS ASTSS SALTED SNACKS ASS...,$1307.18,182


##  Parsing the Sales Data

In [ ]:
sales = pd.DataFrame()
sales['Week Ending'] = pd.to_datetime(raw['Time'].str.replace('1 Week Ending ',''))
sales['Description'] = raw['Product'].str.lower()
sales['Dollar Sales'] = pd.to_numeric(raw['Dollar Sales'].str.replace('$',''))
sales['Unit Sales'] = pd.to_numeric(raw['Unit Sales'])
sales.sort_values(
    by=['Week Ending', 'Dollar Sales'],
    ascending=[True, False],
    inplace=True
)
sales.reset_index(drop=True,inplace=True)
print(sales)


      Week Ending                                        Description  \
0      2017-12-31  doritos tortilla chip nacho cheese bag 9.75 oz...   
1      2017-12-31  lays potato chip classic bag 10 oz - 002840064...   
2      2017-12-31  ruffles potato chip original zero grams trans ...   
3      2017-12-31  cheetos cheese snack flamin hot bag 8.5 oz - 0...   
4      2017-12-31  ruffles potato chip original bag 9 oz - 002840...   
...           ...                                                ...   
14124  2018-12-23  cheetos cheese snack flamin hot bag 2.625 oz -...   
14125  2018-12-23  cheetos cheese snack chipotle ranch bag 3.5 oz...   
14126  2018-12-23  fritos corn chip chili cheese bag 1.125 oz - 0...   
14127  2018-12-23  funyuns onion ring onion plastic bag .75 oz - ...   
14128  2018-12-23  lays potato chip barbecue bag 1 oz - 002840000...   

       Dollar Sales  Unit Sales  
0          51815.92       18555  
1          41606.28       15568  
2          38568.14        9766  

## Parsing Packaging Information and Obtaining Its Prefix

In [ ]:
sales['Packaging'] = sales['Description'].str.extract(r'(bag|assorted|canister)')
prefix = sales['Description'].str.split(r'(bag|assorted|canister)').str[0].str.strip()
prefix = prefix.str.replace('plastic','').str.replace('resealable','').str.strip()

In [ ]:
sales['Packaging'].value_counts()

Packaging
bag         13087
assorted      824
canister      218
Name: count, dtype: int64

In [7]:
# Test code 2
prefix.value_counts()

Description
all frito lay products astss salted snacks    618
cheetos cheese snack cheese                   436
cheetos cheese snack flamin hot               301
doritos tortilla chip nacho cheese            272
lays potato chip classic                      263
                                             ... 
smartfood rte popcorn movie theater butter      1
doritos potato crisp blazin buffalo             1
frito lay variety pack fiery mix                1
doritos tortilla chip toasted corn              1
stacys cheese snack romano & garlic pppr        1
Name: count, Length: 245, dtype: int64

In [8]:
# Test code 3
prefix.str.split().str[-1].unique()

array(['cheese', 'classic', 'fat', 'hot', 'original', 'cream', 'ranch',
       'barbeque', 'limon', 'snacks', 'nacho', 'onion', 'regular',
       'salted', 'naked', 'cheddar', 'jalapeno', 'vinegar', 'chili',
       'barbecue', 'verde', 'tapatio', 'salsa', 'dressed', 'herb', 'bbq',
       'traditional', 'salt', 'free', 'spicy', 'skn', 'flamas', 'taco',
       'grain', 'fix', 'mix', 'cinnamon', 'jalapen', 'chees', 'popcorn',
       'everything', 'explosion', 'potato', 'peanut', 'pretzel', 'garlic',
       'medley', 'prtzl', 'jalapn', 'chs', 'sugar', 'parmsn', 'lime',
       'cheddr', 'oil', 'thymed', 'dijon', 'bread', 'wing', 'cookie',
       'waffles', 'lemon', 'sauce', 'wasabi', 'wngs', 'buffalo', 'queso',
       'blaze', 'pepper', 'butter', 'gravy', 'marinar', 'ranchera', 'sgr',
       'chipotle', 'caramel', 'veggie', 'pppr', 'coconut', 'brbc', 'corn',
       'adobadas', 'frs', 'fries', 'pickle', 'rosemary'], dtype=object)

## Parsing Brand and Flavor Information from the Prefix

In [ ]:
pattern = r'(chips?|snacks?|crisp|pack|skin|ring|cracker|organic|chicharrones)'
sales['Flavor'] = prefix.str.split(pattern).str[-1]
sales['Product'] = prefix.str.extract(r'(.+)(chips?|snacks?|crisp|pack|skin|ring|cracker|organic|chicharrones)')[0]


In [ ]:
# Test code
sales[['Week Ending','Product', 'Flavor', 'Description']]

,Week Ending,Product,Flavor,Description
0,2017-12-31,doritos tortilla,nacho cheese,doritos tortilla chip nacho cheese bag 9.75 oz...
1,2017-12-31,lays potato,classic,lays potato chip classic bag 10 oz - 002840064...
2,2017-12-31,ruffles potato,original zero grams trans fat,ruffles potato chip original zero grams trans ...
3,2017-12-31,cheetos cheese,flamin hot,cheetos cheese snack flamin hot bag 8.5 oz - 0...
4,2017-12-31,ruffles potato,original,ruffles potato chip original bag 9 oz - 002840...
...,...,...,...,...
14124,2018-12-23,cheetos cheese,flamin hot,cheetos cheese snack flamin hot bag 2.625 oz -...
14125,2018-12-23,cheetos cheese snack,otle ranch,cheetos cheese snack chipotle ranch bag 3.5 oz...
14126,2018-12-23,fritos corn,chili cheese,fritos corn chip chili cheese bag 1.125 oz - 0...
14127,2018-12-23,funyuns onion,onion,funyuns onion ring onion plastic bag .75 oz - ...


## Parsing Remaining Attributes from Sales Data

In [ ]:
sales['Product Code'] = pd.to_numeric(sales['Description'].str.split('-').str[-1].str.strip())
sales['Oz'] = sales['Description'].str.extract(r'(\d*\.?\d+)\s*oz').astype(float)
sales['Price Per Oz'] = sales['Dollar Sales'] / sales['Unit Sales'] / sales['Oz']

In [ ]:
# Test code 1
sales.to_csv('11-cleaned_sales.csv.gz', compression='gzip', index=False)
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14129 entries, 0 to 14128
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Week Ending   14129 non-null  datetime64[ns]
 1   Description   14129 non-null  object        
 2   Dollar Sales  14129 non-null  float64       
 3   Unit Sales    14129 non-null  int64         
 4   Packaging     14129 non-null  object        
 5   Flavor        14129 non-null  object        
 6   Product       13024 non-null  object        
 7   Product Code  14129 non-null  int64         
 8   Oz            14078 non-null  float64       
 9   Price Per Oz  14078 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 1.1+ MB


,Week Ending,Description,Dollar Sales,Unit Sales,Packaging,Flavor,Product,Product Code,Oz,Price Per Oz
0,2017-12-31,doritos tortilla chip nacho cheese bag 9.75 oz...,51815.92,18555,bag,nacho cheese,doritos tortilla,28400642031,9.750,0.286416
1,2017-12-31,lays potato chip classic bag 10 oz - 002840064...,41606.28,15568,bag,classic,lays potato,28400645491,10.000,0.267255
2,2017-12-31,ruffles potato chip original zero grams trans ...,38568.14,9766,bag,original zero grams trans fat,ruffles potato,28400034001,13.500,0.292535
3,2017-12-31,cheetos cheese snack flamin hot bag 8.5 oz - 0...,33648.16,11013,bag,flamin hot,cheetos cheese,28400589891,8.500,0.359449
4,2017-12-31,ruffles potato chip original bag 9 oz - 002840...,30916.92,10519,bag,original,ruffles potato,28400159381,9.000,0.326572
...,...,...,...,...,...,...,...,...,...,...
14124,2018-12-23,cheetos cheese snack flamin hot bag 2.625 oz -...,1.89,1,bag,flamin hot,cheetos cheese,28400624431,2.625,0.720000
14125,2018-12-23,cheetos cheese snack chipotle ranch bag 3.5 oz...,1.89,1,bag,otle ranch,cheetos cheese snack,28400648761,3.500,0.540000
14126,2018-12-23,fritos corn chip chili cheese bag 1.125 oz - 0...,0.99,3,bag,chili cheese,fritos corn,28400161851,1.125,0.293333
14127,2018-12-23,funyuns onion ring onion plastic bag .75 oz - ...,0.99,3,bag,onion,funyuns onion,28400090842,0.750,0.440000


In [23]:
# Test code 2
sales[['Week Ending','Product','Flavor','Packaging','Oz','Product Code','Dollar Sales','Price Per Oz']]

,Week Ending,Product,Flavor,Packaging,Oz,Product Code,Dollar Sales,Price Per Oz
0,2017-12-31,doritos tortilla,nacho cheese,bag,9.750,28400642031,51815.92,0.286416
1,2017-12-31,lays potato,classic,bag,10.000,28400645491,41606.28,0.267255
2,2017-12-31,ruffles potato,original zero grams trans fat,bag,13.500,28400034001,38568.14,0.292535
3,2017-12-31,cheetos cheese,flamin hot,bag,8.500,28400589891,33648.16,0.359449
4,2017-12-31,ruffles potato,original,bag,9.000,28400159381,30916.92,0.326572
...,...,...,...,...,...,...,...,...
14124,2018-12-23,cheetos cheese,flamin hot,bag,2.625,28400624431,1.89,0.720000
14125,2018-12-23,cheetos cheese snack,otle ranch,bag,3.500,28400648761,1.89,0.540000
14126,2018-12-23,fritos corn,chili cheese,bag,1.125,28400161851,0.99,0.293333
14127,2018-12-23,funyuns onion,onion,bag,0.750,28400090842,0.99,0.440000


## Identifying Most Common Phrases in Product Descriptions

In [ ]:
def common_phrases(strings, min_occurrence, inclusion_ratio=1):
    import pandas as pd
    
    phrase_counts = {}
    
    for string in strings:
        words = string.lower().split()
        for length in range(1, len(words) + 1):
            for i in range(len(words) - length + 1):
                phrase = ' '.join(words[i:i+length])
                phrase_counts[phrase] = phrase_counts.get(phrase, 0) + 1
    
    filtered = {p: c for p, c in phrase_counts.items() if c >= min_occurrence}
    
    result = {}
    for phrase, count in filtered.items():
        keep = True

        for other_phrase, other_count in filtered.items():
            if phrase != other_phrase and phrase in other_phrase:
                if other_count >= count * inclusion_ratio:
                    keep = False
                    break
        if keep:
            result[phrase] = count
    
    return pd.Series(result).sort_values(ascending=False)


In [33]:
# Test code 1
strings=['apple', 'apple pie', 'pumpkin pie', 'tasty apple pie', 'apple crisp', 'pie crust', 'chocolate pie', 'chocolate pie crust', 'tasty chocolate pie']
print(common_phrases(strings, 2))

pie              7
apple            4
chocolate pie    3
apple pie        2
tasty            2
pie crust        2
dtype: int64


In [34]:
# Test code 2 (creates the 11-keywords.csv file based on common phrases in product descriptions)
strings=sales['Description'].str.replace(r'plastic|bag|oz|[^a-z\s]','',regex=True).unique()
keywords=common_phrases(strings, 7, 1.1)
keywords.to_csv('11-keywords.csv')
print(keywords.index)

Index(['chip', 'potato', 'potato chip', 'lays', 'tortilla', 'tortilla chip',
       'snack', 'cheese', 'lays potato chip', 'lays potato',
       ...
       'mix', 'astss salted', 'sweet', 'barbecue', 'fritos corn chip',
       'fritos corn', 'astss salted snacks', 'astss', 'snacks', 'chesters'],
      dtype='object', length=116)
